# Reliability Analysis Using ICC
*Suraj Puvvadi*

*04-29-26*

This notebook calculates the **reliability of arm metrics** across repeated trials using the **Intraclass Correlation Coefficient (ICC)**, implemented via the `pingouin` open-source Python package.  

**Key points:**  
- **Data:** Metrics are read from a CSV file containing multiple subjects, trial numbers, and test types (`RG` and `NG`)
- **ICC Model:** Follows Koo & Li (2015) recommendations:  
  - Two-way mixed-effects model (`ICC(3,1)`) for **absolute agreement**  
  - Suitable for **single-measure reliability** across repeated trials  
  - Captures variability from both subjects and trials
  - **Goal**: Measure reliability of digital biomarkers across repeated trials without interventions
- **Processing:**  
  - Calculates ICC for each metric separately  
  - Handles missing data by dropping incomplete subject-trial combinations  
- **Visualization:**  
  - Horizontal bar plots display ICC values per metric  
  - Positive values in blue, negative in orange  
  - Annotated with exact ICC numbers for clarity  

**Outputs:**  
- ICC values for each metric and test type (`RG` and `NG`)  
- High-resolution bar plots saved to a specified output folder

**References:**  
- Pingouin package ICC documentation, https://pingouin-stats.org/generated/pingouin.intraclass_corr.html
- Koo & Li, *Journal of Chiropractic Medicine*, 2015, https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4913118/

In [ ]:
# ICC WITH CONFIDENCE INTERVALS (FIXED VERSION)

import os
import pandas as pd
import pingouin as pg
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("ggplot")

# =====================================================================
# INPUT / OUTPUT
# =====================================================================
input_folder = "/Users/suraj/Desktop/daad/code/finalized_programs/arm_analysis_results_20260405"
output_folder = "/Users/suraj/Desktop/daad/code/finalized_programs/final_icc_plots_20260405"
os.makedirs(output_folder, exist_ok=True)

csv_files = [
    f for f in os.listdir(input_folder)
    if "avg_arm_analysis_summary" in f and f.endswith(".csv")
]

print("Found CSVs:", csv_files)

# =====================================================================
# ICC WITH CI
# =====================================================================

def compute_icc(df, metric_cols, test_type):
    results = {}

    subset = df[df["test_type"] == test_type]

    for col in metric_cols:
        data = subset[["subject", "trial_num", col]].pivot(
            index="subject",
            columns="trial_num",
            values=col
        ).dropna()

        # need at least 2 subjects
        if data.shape[0] <= 1:
            continue

        melted = data.reset_index().melt(
            id_vars="subject",
            var_name="trial_num",
            value_name=col
        )

        melted["trial_num"] = melted["trial_num"].astype(int)

        icc_table = pg.intraclass_corr(
            data=melted,
            targets="subject",
            raters="trial_num",
            ratings=col
        ).set_index("Type")

        # Prefer ICC3 if available
        if "ICC3" in icc_table.index:
            row = icc_table.loc["ICC3"]
        else:
            row = icc_table.iloc[0]

        icc_val = row["ICC"]

        # SAFE CI extraction (Pingouin sometimes returns tuple or Series)
        ci = row.get("CI95%", (np.nan, np.nan))
        try:
            ci_low, ci_high = ci
        except Exception:
            ci_low, ci_high = np.nan, np.nan

        results[col] = {
            "icc": icc_val,
            "low": ci_low,
            "high": ci_high
        }

    return results

# =====================================================================
# PLOTTING
# =====================================================================

def plot_top10_icc(ax, icc_dict, title, color):
    if not icc_dict:
        ax.set_title(title + "\n(NO DATA)")
        ax.axis("off")
        return

    df = pd.DataFrame.from_dict(icc_dict, orient="index").reset_index()
    df = df.rename(columns={"index": "metric"})

    df = df.sort_values("icc", ascending=False).head(10).sort_values("icc")

    y = np.arange(len(df))

    ax.barh(y, df["icc"], color=color)

    # CI error bars
    ax.errorbar(
        df["icc"], y,
        xerr=[df["icc"] - df["low"], df["high"] - df["icc"]],
        fmt="none",
        ecolor="black",
        capsize=3,
        linewidth=1
    )

    ax.set_yticks(y)
    ax.set_yticklabels(df["metric"], fontsize=9, fontweight="bold")

    ax.set_xlim(0, 1.1)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.grid(True, axis="x", linestyle="--", alpha=0.4)

    # fixed x-position column for ALL ICC labels
    label_x = 1.05  # choose a constant column position inside axis

    for i, v in enumerate(df["icc"]):
        ax.text(label_x, i, f"{v:.2f}",
                va="center",
                ha="right",
                fontsize=9,
                fontweight="bold")

# =====================================================================
# MAIN
# =====================================================================

results = {
    "Ataxia_NG": None,
    "PD_NG": None,
    "Ataxia_RG": None,
    "PD_RG": None
}

for file in csv_files:
    file_path = os.path.join(input_folder, file)
    print("Processing:", file)

    df = pd.read_csv(file_path)
    metric_cols = df.columns[3:]
    fname = file.lower()

    if "ataxia" in fname:
        key = "Ataxia"
    elif "pd" in fname:
        key = "PD"
    else:
        continue

    print(f"  → Group: {key}")

    results[f"{key}_NG"] = compute_icc(df, metric_cols, "NG")
    results[f"{key}_RG"] = compute_icc(df, metric_cols, "RG")

# =====================================================================
# PLOT GRID
# =====================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

plot_top10_icc(axes[0, 0], results["Ataxia_NG"], "Ataxia – NG", "steelblue")
plot_top10_icc(axes[0, 1], results["PD_NG"], "PD – NG", "steelblue")
plot_top10_icc(axes[1, 0], results["Ataxia_RG"], "Ataxia – RG", "#800000")
plot_top10_icc(axes[1, 1], results["PD_RG"], "PD – RG", "#800000")

plt.tight_layout()

outfile = os.path.join(output_folder, "ICC_top10_with_CI_matrix_fixed.png")
plt.savefig(outfile, dpi=300)
plt.close()

print("Saved:", outfile)

Found CSVs: ['ataxia_avg_arm_analysis_summary.csv', 'pd_avg_arm_analysis_summary.csv']
Processing: ataxia_avg_arm_analysis_summary.csv
  → Group: Ataxia
Processing: pd_avg_arm_analysis_summary.csv
  → Group: PD
Saved: /Users/suraj/Desktop/daad/code/finalized_programs/final_icc_plots_20260405/ICC_top10_with_CI_matrix_fixed.png
